# Exercice 4 - Expected Loss et CDO

CDO avec 125 noms, maturité 1 an, coupon 1 unité, recovery = 0.

3 tranches : equity, mezzanine, senior. Modele de Vasicek (corrélation identique).

In [ ]:
import numpy as np
from scipy.stats import norm, binom
from scipy.integrate import quad
import matplotlib.pyplot as plt

## Question 1 - Proba de défaut conditionnelle

Dans Vasicek, l'actif i fait défaut si :
$$X_i = \sqrt{\rho} F + \sqrt{1-\rho} \varepsilon_i < \Phi^{-1}(PD)$$

Conditionnellement à F=f :
$$p(f) = P(def_i | F=f) = \Phi\left(\frac{\Phi^{-1}(PD) - \sqrt{\rho} f}{\sqrt{1-\rho}}\right)$$

Les défauts sont indépendants conditionnellement à F.

In [ ]:
def p_cond(f, PD, rho):
    """proba defaut conditionnelle au facteur f"""
    return norm.cdf((norm.ppf(PD) - np.sqrt(rho)*f) / np.sqrt(1-rho))

## Question 2 - Loi binomiale et proba non conditionnelle

Sachant F=f, le nb de défauts K suit une binomiale :
$$K | F=f \sim Bin(N, p(f))$$

On integre sur f pour avoir la proba non conditionnelle :
$$P(K=k) = \int_{-\infty}^{+\infty} C_N^k \cdot p(f)^k (1-p(f))^{N-k} \cdot \phi(f) df$$

In [ ]:
N = 125
PD = 0.02  # 2%
rho = 0.10

In [ ]:
def proba_k(k, N, PD, rho):
    """P(K=k) par intégration sur le facteur"""
    def integrande(f):
        pc = p_cond(f, PD, rho)
        return binom.pmf(k, N, pc) * norm.pdf(f)
    res, _ = quad(integrande, -5, 5)
    return res

In [ ]:
# on calcule toute la distribution
print("Calcul de la distribution...")
dist = np.array([proba_k(k, N, PD, rho) for k in range(N+1)])

print(f"Somme des probas = {dist.sum():.4f}")
print(f"E[K] = {sum(k*dist[k] for k in range(N+1)):.2f} (theorique : {N*PD:.2f})")

## Question 3 - EL par tranche

On definit les tranches par les points d'attachement/detachement en % du pool (soit en nb de defauts) :
- Equity : 0% à 3%  soit 0 à 3.75 → arrondi à [0, 4]
- Mezz : 3% à 7% → [4, 9]
- Senior : 7% à 100% → [9, 125]

La perte de la tranche [Ka, Kd] quand il y a k defauts :
$$L_{tranche}(k) = \min(\max(k - K_a, 0),\; K_d - K_a)$$

In [ ]:
# points d'attachement (en nb de defauts)
Ka_eq, Kd_eq = 0, 4       # equity
Ka_mz, Kd_mz = 4, 9       # mezzanine  
Ka_sr, Kd_sr = 9, 125     # senior

In [ ]:
def EL_tranche(Ka, Kd, distribution):
    taille = Kd - Ka
    el = 0
    for k in range(len(distribution)):
        loss = min(max(k - Ka, 0), taille)
        el += loss * distribution[k]
    return el

el_eq = EL_tranche(Ka_eq, Kd_eq, dist)
el_mz = EL_tranche(Ka_mz, Kd_mz, dist)
el_sr = EL_tranche(Ka_sr, Kd_sr, dist)

print(f"{'Tranche':<12} {'EL (noms)':<12} {'EL (%)':<10}")
print("-"*34)
print(f"{'Equity':<12} {el_eq:<12.4f} {el_eq/(Kd_eq-Ka_eq)*100:<10.2f}")
print(f"{'Mezzanine':<12} {el_mz:<12.4f} {el_mz/(Kd_mz-Ka_mz)*100:<10.2f}")
print(f"{'Senior':<12} {el_sr:<12.6f} {el_sr/(Kd_sr-Ka_sr)*100:<10.4f}")
print(f"\nEL totale = {el_eq + el_mz + el_sr:.4f} (attendu N*PD = {N*PD:.2f})")

In [ ]:
# graphique
fig, ax = plt.subplots(figsize=(9, 4))
k_plot = np.arange(0, 20)
ax.bar(k_plot, dist[:20], color='steelblue', alpha=0.7)
ax.axvline(Kd_eq, color='red', ls='--', label='Détach equity')
ax.axvline(Kd_mz, color='orange', ls='--', label='Détach mezz')
ax.set_xlabel('Nombre de défauts')
ax.set_ylabel('Probabilité')
ax.set_title(f'Distribution des défauts (N={N}, PD={PD*100}%, ρ={rho})')
ax.legend()
plt.tight_layout()
plt.show()

## Sensibilité à la corrélation

La corrélation joue un role crucial dans la redistribution des pertes entre tranches.

In [ ]:
print(f"{'rho':<6} {'EL Eq%':<10} {'EL Mz%':<10} {'EL Sr%':<10}")
print("-"*36)
for rho_test in [0.02, 0.05, 0.10, 0.20, 0.30]:
    d = np.array([proba_k(k, N, PD, rho_test) for k in range(N+1)])
    e1 = EL_tranche(Ka_eq, Kd_eq, d) / (Kd_eq - Ka_eq) * 100
    e2 = EL_tranche(Ka_mz, Kd_mz, d) / (Kd_mz - Ka_mz) * 100
    e3 = EL_tranche(Ka_sr, Kd_sr, d) / (Kd_sr - Ka_sr) * 100
    print(f"{rho_test:<6.2f} {e1:<10.2f} {e2:<10.2f} {e3:<10.4f}")

### Interprétation

La tranche equity absorbe l'essentiel des pertes attendues : son EL est tres élevé. La tranche senior a un EL quasi nul en conditions normales.

Quand ρ augmente : la distribution s'aplatit (queues plus épaisses). L'equity perd moins en moyenne (plus de chances de 0 defaut) mais les tranches senior et mezz perdent plus (scenarios extrêmes plus probables). C'est ce qu'on appelle le "correlation trade" sur les CDOs.